In [ ]:
import pandas as pd
import numpy as np
import ast
from collections import Counter

In [ ]:
df = pd.read_excel("PERCEPT.xlsx")
df.head()

,Text,Source,Topic,Platform,Words,POS,Positions,To Check,Result
0,سلام،اگر دیجی کالا محصولاتی مثل lexar professi...,MirasOpinion,برند و کسب‌وکار,Digikala,"['lexar', 'professional', '256GB', '540mb\\s',...","['PROPN', 'PROPN', 'NUM', 'SYM', 'NOUN']","['initial', 'initial', 'initial', 'medial', 'm...",NaN,"[{'word': 'lexar', 'upos': 'PROPN'}, {'word': ..."
1,با سلام خدمت دوستان من خودم M9 دارم و این یه چ...,MirasOpinion,برند و کسب‌وکار,Digikala,['M9'],['PROPN'],['initial'],NaN,"[{'word': 'M9', 'upos': 'PROPN'}]"
2,سلام دوستان.... من اینو خریدم چند ماه پیش ......,MirasOpinion,برند و کسب‌وکار,Digikala,"['vob', 'mpeg', 'هنگ']","['NOUN', 'NOUN', 'VERB']","['medial', 'medial', 'medial']",NaN,"[{'word': 'vob', 'upos': 'NOUN'}, {'word': 'mp..."
3,سلام این محصول رو امروز تهیه کردم البته به ع...,MirasOpinion,برند و کسب‌وکار,Digikala,"['wifi', 'hp', 'Device', 'Manager', 'Advanced'...","['NOUN', 'PROPN', 'NOUN', 'NOUN', 'ADJ', 'NOUN...","['medial', 'medial', 'medial', 'final', 'final...",NaN,"[{'word': 'wifi', 'upos': 'NOUN'}, {'word': 'h..."
4,متاسفانه برنامه ی دستگاه کاملا چینی هست که یه ...,MirasOpinion,برند و کسب‌وکار,Digikala,['اکانت'],['NOUN'],['initial'],NaN,"[{'word': 'اکانت', 'upos': 'NOUN'}]"


In [ ]:
df['Words'] = df['Words'].apply(ast.literal_eval)

In [ ]:
count = 0

for index, row in df.iterrows():
  if len(row["Words"]) > 1:
    count += 1

count / len(df)

0.29911764705882354

In [ ]:
count

2034

In [ ]:
df["text_length"] = df["Text"].apply(lambda x: len(x.split()))
df["triggering"] = df["Words"].apply(lambda x: len(x) > 1)

df["length_bin"] = pd.cut(
    df["text_length"],
    bins=[0, 10, 30, float("inf")],
    labels=["Short (≤10 words)", "Medium (11–30 words)", "Long (>30 words)"]
)

triggering_by_length = (
    df.groupby(["Platform", "length_bin"], observed=True)
      .agg(
          utterances=("triggering", "count"),
          multiple_cm_utterances=("triggering", "sum"),
          triggering_rate=("triggering", "mean")
      )
      .reset_index()
)

triggering_by_length["triggering_rate"] *= 100
triggering_by_length

In [ ]:
length_controlled_triggering = (
    triggering_by_length
    .groupby("Platform", observed=True)["triggering_rate"]
    .mean()
    .reset_index()
    .rename(columns={"triggering_rate": "length_controlled_triggering"})
)

length_controlled_triggering

In [ ]:
results = {}

for platform, df_p in df.groupby("Platform"):
    count = 0

    for _, row in df_p.iterrows():
        if len(row["Words"]) > 1:
            count += 1

    results[platform] = count
results

{'Digikala': 747, 'Instagram': 148, 'Twitter': 1139}